In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# names.txt is downloaded from https://github.com/karpathy/makemore/blob/master/names.txt
words = open('../data/makemore/names.txt', 'r').read().splitlines()
words[:8]

In [ ]:
len(words)

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

In [ ]:
block_size = 3  # context length

def build_dataset(words):
    X, Y = [], []
    for w in words:  # Or set the [:5] to look at the examples and check mini-batch overfitting
        # print(w)
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '--->', itos[ix])
            context = context[1:] + [ix]  # Shift the context window

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])      # 80%
Xdev, Ydev = build_dataset(words[n1:n2])  # 10%
Xte, Yte = build_dataset(words[n2:])      # 10%

In [ ]:
n_embd = 10
n_hidded = 200

# Defining the parameters
# Remember, we need to initialize the parameters of a NN so that the signal going through this NN looks lika a Gaussian
# Additionally, we want to make sure, that our nonlinearities (tanh in this case) work fine (signal is neither vanished, nor saturated)
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidded), generator=g) * 0.2  # 2) Making the output closer to 0 to reduce tanh saturation (fixing the tanh)
# b1 = torch.randn(n_hidded, generator=g) * 0.01  # 2) Making the output closer to 0 to reduce tanh saturation (fixing the tanh)
W2 = torch.randn((n_hidded, vocab_size), generator=g) * 0.01  # 1) Making the output layer weights more uniformly distributied (fixing the loss)
b2 = torch.randn(vocab_size, generator=g) * 0.0  # 1) Making the output layer weights more uniformly distributied (fixing the loss)

# The ratios during the initializaation is a dummy way to make the distributions for this NN more stable:
# 1) Uniform distribution of the output logits
# 2) Close to zero weights as an input for tanh to reduce its saturation
# But there is another problem: distribution tails becomes larger after matrix multiplication (the distribution is squizes - std increases)
# To mitigate this we will need to multiply the initial weights by a coefficient or use torch.nn.init special functions
# (e.g., they based on the Kaiming etc. paper).
# The new initialization of the W1 is presented bellow

W1 = torch.randn((n_embd * block_size, n_hidded), generator=g) * (5/3) / (n_embd * block_size ** 0.5)  # The Kaiming init (https://docs.pytorch.org/docs/2.9/nn.init.html)

# Batch Norm parameters
bngain = torch.ones((1, n_hidded))  # Initialize to not affect the hpreact (the value will be learned during the backprop)
bnbias = torch.zeros((1, n_hidded))  # Initialize to not affect the hpreact (the value will be learned during the backprop)
bnmean_running = torch.zeros((1, n_hidded))  # Initialize with the Gaussian value
bnstd_running = torch.ones((1, n_hidded))  # Initialize with the Gaussian value

# parameters = [C, W1, b1, W2, b2, bngain, bnbias]
parameters = [C, W1, W2, b2, bngain, bnbias]  # Without b1
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
    # Mini-batch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))  # Batch size is batch_size
    Xb, Yb = Xtr[ix], Ytr[ix]

    # Forward pass
    emb = C[Xb]  # Embed the characters into vectors
    embcat = emb.view(emb.shape[0], -1)  # Concatenate the vectors
    hpreact = embcat @ W1 #+ b1  # Hidden layer pre-activation (if you use batch norm, then b1 is useless)

    # ----- Batch normalization -----
    # Use Batch normalization after matrix multiplications, to normalize distributions after
    # The distributions are usually squized (std becomes higher) after matrix multiplication,
    # which leads to saturation of the activation function (tanh is this example)
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani  # An approximation (EMA: 0.001 is a momentum)
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi  # An approximation (EMA: 0.001 is a momentum)
    # ----- / -----

    h = torch.tanh(hpreact)  # Hidden layer
    logits = h @ W2 + b2  # Output layer
    loss = F.cross_entropy(logits, Ytr[ix])

    # Backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # Parameters update
    lr = 0.1 if i < 100000 else 0.01  # Step learning rate decay
    for p in parameters:
        p.data += -lr * p.grad

    # Track stats
    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

    # break

print(f'{max_steps}/{max_steps}: {loss.item():.4f}')

Fixing the loss by making the output logits distribution more nuiform like

In [ ]:
# # The initial loss we want to expect
# # But initializing the weights with normal distribution we have a way higher value: 25.55
# # We decided to scale weights by 0.01 (* 0.01) and make biases equal to 0 (* 0.0)
# -torch.tensor(1/27.0).log()

Fixing the saturated tanh function

In [ ]:
# # tanh output distribution
# plt.hist(h.view(-1).tolist(), bins=50)

In [ ]:
# # tanh input distribution
# plt.hist(hpreact.view(-1).tolist(), bins=50)

In [ ]:
# # Looking at the binary map of the tanh activations for a minibatch after the initialization
# # Too many white pixels tells that the gradient will be vanished during the back propogation
# # (tanh values are too high: [-1; 1 - tanh is saturated]), because the gradient will be zero
# plt.figure(figsize=(20, 10))
# plt.imshow(h.abs() > 0.99, cmap='gray', interpolation='nearest')

In [ ]:
plt.plot(lossi)

The loss is better and it doesn't look like a 'hokey stick'

In [ ]:
@torch.no_grad()  # Do not require the grad for the only forward pass (for efficiency)
def split_loss(split):
    x, y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    emb = C[x]  # (N, block_size, n_embd)
    embcat = emb.view(emb.shape[0], -1)  # Concat into (N, block_size * n_embd)
    hpreact = embcat @ W1 #+ b1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h = torch.tanh(hpreact)  # (N, n_hidden)
    logits = h @ W2 + b2  # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

In [ ]:
# Sampling
inf_gen = torch.Generator().manual_seed(2147483647 + 10)
for i in range(20):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        embcat = emb.view(emb.shape[0], -1)
        hpreact = embcat @ W1 #+ b1
        hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2
        probs = F.softmax(logits, 1)

        idx = torch.multinomial(probs, num_samples=1, replacement=True, generator=inf_gen).item()
        context = context[1:] + [idx]
        out.append(itos[idx])
        if idx == 0:
            break

    print(''.join(out).strip('.'))

Creating a little bit deeper NN

In [ ]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5  # Torch like init
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([self.bias] if self.bias is not None else [])


class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        # Trainable parameters
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        # Running statistics (buffers)
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim=True)  # Batch mean
            xvar = x.var(0, keepdim=True)  # Batch variance
        else:
            xmean = self.running_mean
            xvar = self.running_var

        xhat = (x - xmean) / (torch.sqrt(xvar) + self.eps)  # Normalize to unit variance
        self.out = self.gamma * xhat + self.beta

        # Update the running statistics (buffers)
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar

        return self.out


    def parameters(self):
        return [self.gamma, self.beta]


class Tanh:
    def __init__(self):
        pass

    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []


# The initialization
n_embd = 10
n_hidded = 100
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embd), generator=g)
# We add the tanh layres as a nonlinearity to ensure, that the model can approximate an arbitrary function
# Without this nonlinearity all the linear layres rawly collapse to a one linear layer and can approximate only linear dependencies
layers = [
    Linear(n_embd * block_size, n_hidded), BatchNorm1d(n_hidded), Tanh(),
    Linear(n_hidded, n_hidded), BatchNorm1d(n_hidded), Tanh(),
    Linear(n_hidded, n_hidded), BatchNorm1d(n_hidded), Tanh(),
    Linear(n_hidded, n_hidded), BatchNorm1d(n_hidded), Tanh(),
    Linear(n_hidded, n_hidded), BatchNorm1d(n_hidded), Tanh(),
    Linear(n_hidded, vocab_size), BatchNorm1d(vocab_size),
]

with torch.no_grad():
    # Correct the initial weights of the output layer
    # layers[-1].weight *= 0.1
    layers[-1].gamma *= 0.1
    # Apply gain for all other layers according to the Kaiming init (https://docs.pytorch.org/docs/2.9/nn.init.html)
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.weight *= 5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
max_steps = 200000
batch_size = 32
lossi = []
ud = []  # Updates to data tracking

for i in range(max_steps):
    # Mini-batch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)  # Batch size is batch_size
    Xb, Yb = Xtr[ix], Ytr[ix]

    # Forward pass
    emb = C[Xb]  # Embed the characters into vectors
    x = emb.view(emb.shape[0], -1)  # Concatenate the vectors
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x, Yb)

    # Backward pass
    for layer in layers:
        layer.out.retain_grad()  # ?
    for p in parameters:
        p.grad = None
    loss.backward()

    # Parameters update
    lr = 0.1 if i < 100000 else 0.01  # Step learning rate decay
    for p in parameters:
        p.data += -lr * p.grad

    # Track stats
    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

    with torch.no_grad():
        ud.append([(lr*p.grad.std() / p.data.std()).log10().item() for p in parameters])

    if i >= 1000:
        break

print(f'{max_steps}/{max_steps}: {loss.item():.4f}')

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('activation distribution')

The `tanh` activations on the 0-th iteration of the training shouldn't be too saturated. The gain equal to `5/3` helps to prevent it.

Set this gain to something higher and you'll see the higher saturation, which leads to vanishing gradients. Set the gain to e.g. 1 and you'll see that the values on the more far layers are squizing, which isn't good either - almost no signal, only noise.

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('gradient distribution')

The gradients of `tanh` layers on the first iteration of the training have rawly normal distribution and this distribution is almost the same for all the layres.

The gradients start changing if we change the `5/3` gain for the layres initialization with more differences depending on the level of the layer.

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, p in enumerate(parameters):
  t = p.grad
  if p.ndim == 2:  # Only linear layers
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'{i} {tuple(p.shape)}')
plt.legend(legends)
plt.title('weights gradient distribution')

The last layer gradient looks different from all other layres (on the first iteration, and even on the 1000-th iteration), but during the training it is fixing itself little by little.

In [ ]:
plt.figure(figsize=(20, 4))
legends = []
for i, p in enumerate(parameters):
  if p.ndim == 2:
    plt.plot([ud[j][i] for j in range(len(ud))])
    legends.append('param %d' % i)
plt.plot([0, len(ud)], [-3, -3], 'k') # these ratios should be ~1e-3, indicate on plot
plt.legend(legends)

The plot above (updates dynamic) can show the training speed/efficiency. The layers normally should be trainged almost the same (except for the first and the last layer) and the `grad.std / data.std` should be stabilized over the training iterations.

This plot also can help with detemine the learning_rate: we can see how fast the NN is training (is it far bellow the 1e-3, or far above).